# Complete Flight Operations Data Analysis

Download the Flight Operations CSV from the LMS **Study Material** tab. Run the single code cell below and upload the file when prompted.

In [ ]:
# Complete Flight Operations analysis — single Google Colab cell
import pandas as pd
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Flight Operations CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip()
pd.set_option('display.max_columns', None)

# Finds common field-name variants without requiring an exact CSV schema.
def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

airline_col = find_column(['airline', 'carrier', 'operator'])
origin_col = find_column(['origin', 'departure airport', 'source airport', 'from'])
destination_col = find_column(['destination', 'arrival airport', 'destination airport', 'to'])
status_col = find_column(['status', 'flight status'])
delay_col = find_column(['delay minutes', 'delay', 'departure delay', 'arrival delay'])
date_col = find_column(['flight date', 'date', 'departure date', 'scheduled date'])
flight_col = find_column(['flight number', 'flight no', 'flight id'])

# Convert likely numeric delay values and date values for analysis.
if delay_col:
    df[delay_col] = pd.to_numeric(df[delay_col].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df['Flight_Month'] = df[date_col].dt.month_name()
    df['Flight_Day_of_Week'] = df[date_col].dt.day_name()
if delay_col:
    # apply() transformation: place each flight into a delay category.
    df['Delay_Category'] = df[delay_col].apply(lambda x: 'Unknown' if pd.isna(x) else ('On Time' if x <= 0 else ('Minor Delay' if x <= 30 else 'Major Delay')))

print('=' * 85)
print(f'COMPLETE FLIGHT OPERATIONS DATA ANALYSIS: {csv_files[0]}')
print('=' * 85)

print('\n1. DATA UNDERSTANDING')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns')
print('Columns:', df.columns.tolist())
print('\nFirst five rows:')
display(df.head())
print('\nLast five rows:')
display(df.tail())
print('\nData types:')
display(df.dtypes.to_frame(name='Data Type'))
print('\nDataFrame information:')
df.info()
print('\nDescriptive statistics:')
display(df.describe(include='all').T)

print('\n2. DATA QUALITY CHECK')
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({'Missing Values': missing, 'Missing (%)': (missing / len(df) * 100).round(2)}))
print(f'Duplicate records: {df.duplicated().sum():,}')

print('\n3. DATA SELECTION, FILTERING, AND SORTING')
key_columns = [column for column in [flight_col, airline_col, origin_col, destination_col, date_col, status_col, delay_col, 'Delay_Category'] if column]
display(df[key_columns].head(10) if key_columns else df.head(10))
if delay_col:
    delayed_flights = df[df[delay_col] > 0].sort_values(delay_col, ascending=False)
    print(f'Flights delayed by more than 0 minutes: {len(delayed_flights):,}; top 10 most delayed flights:')
    display(delayed_flights[key_columns].head(10) if key_columns else delayed_flights.head(10))
else:
    print('No delay field was identified, so delay filtering was skipped.')

def group_summary(column, label):
    if not column:
        print(f'\n{label}: relevant column not found.')
        return None
    aggregations = {'Number_of_Flights': (column, 'size')}
    if delay_col:
        aggregations['Average_Delay'] = (delay_col, 'mean')
        aggregations['Maximum_Delay'] = (delay_col, 'max')
    summary = df.groupby(column).agg(**aggregations).sort_values('Number_of_Flights', ascending=False)
    if delay_col:
        summary[['Average_Delay', 'Maximum_Delay']] = summary[['Average_Delay', 'Maximum_Delay']].round(2)
    print(f'\n{label}')
    display(summary)
    return summary

print('\n4. GROUPING AND AGGREGATION')
airline_summary = group_summary(airline_col, 'Flights and delays by airline')
origin_summary = group_summary(origin_col, 'Flights and delays by origin airport')
destination_summary = group_summary(destination_col, 'Flights and delays by destination airport')
status_summary = group_summary(status_col, 'Flights and delays by status')
if 'Flight_Day_of_Week' in df.columns:
    weekday_summary = group_summary('Flight_Day_of_Week', 'Flights and delays by day of week')
else:
    weekday_summary = None

print('\n5. KEY FINDINGS')
observations = [
    f'1. The dataset contains {len(df):,} flight records across {len(df.columns):,} fields.',
    f'2. It contains {df.select_dtypes(include="number").shape[1]} numeric field(s) and {df.select_dtypes(exclude="number").shape[1]} non-numeric field(s).',
    f'3. There are {df.duplicated().sum():,} duplicate row(s) and {missing.sum():,} total missing value(s).',
]
if delay_col:
    observations.append(f'4. Average delay is {df[delay_col].mean():.2f} minutes; the maximum delay is {df[delay_col].max():.2f} minutes.')
    observations.append(f'5. {int((df[delay_col] > 0).sum()):,} flights ({(df[delay_col] > 0).mean() * 100:.1f}%) recorded a delay above zero minutes.')
if airline_summary is not None:
    observations.append(f'{len(observations) + 1}. The busiest airline/operator is {airline_summary.index[0]} with {airline_summary.iloc[0]["Number_of_Flights"]:,} flights.')
if origin_summary is not None:
    observations.append(f'{len(observations) + 1}. The busiest origin airport is {origin_summary.index[0]} with {origin_summary.iloc[0]["Number_of_Flights"]:,} flights.')
if destination_summary is not None:
    observations.append(f'{len(observations) + 1}. The most frequent destination is {destination_summary.index[0]} with {destination_summary.iloc[0]["Number_of_Flights"]:,} flights.')
if weekday_summary is not None:
    observations.append(f'{len(observations) + 1}. The busiest day of the week is {weekday_summary.index[0]} with {weekday_summary.iloc[0]["Number_of_Flights"]:,} flights.')
for finding in observations[:8]:
    print(finding)